In [ ]:
!pip install -q langgraph langchain easyocr ultralytics opencv-python

In [ ]:
!pip install -q transformers langgraph langchain easyocr ultralytics opencv-python

import cv2
import urllib.request
import numpy as np
import easyocr
import matplotlib.pyplot as plt
from ultralytics import YOLO
from typing import TypedDict, List, Dict
from langgraph.graph import StateGraph, END
from transformers import pipeline

class AgentState(TypedDict):
    image_path: str
    user_query: str
    processed_image: np.ndarray
    detected_regions: List[Dict]
    extracted_text: str
    final_answer: str

reader = easyocr.Reader(['en'], gpu=False)
layout_model = YOLO("yolov8n.pt")
qa_model = pipeline("question-answering", model="deepset/roberta-base-squad2")

def vision_preprocessing_node(state: AgentState):
    img = cv2.imread(state["image_path"])
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    clean_img = cv2.adaptiveThreshold(
        gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 11, 2
    )
    return {"processed_image": clean_img}

def layout_analysis_node(state: AgentState):
    img = state["processed_image"]
    img_rgb = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
    results = layout_model(img_rgb, verbose=False)
    regions = []

    if len(results[0].boxes) > 0:
        boxes = results[0].boxes.xyxy.cpu().numpy()
        boxes = sorted(boxes, key=lambda b: b[1])
        for box in boxes:
            x1, y1, x2, y2 = map(int, box)
            regions.append({"bbox": (x1, y1, x2, y2), "crop": img[y1:y2, x1:x2]})
    else:
        h, w = img.shape
        regions.append({"bbox": (0, 0, w, h), "crop": img})

    return {"detected_regions": regions}

def ocr_extraction_node(state: AgentState):
    regions = state["detected_regions"]
    full_text = []
    for region in regions:
        crop = region["crop"]
        if crop.size > 0:
            result = reader.readtext(crop, detail=0)
            full_text.extend(result)
    return {"extracted_text": " ".join(full_text)}

def llm_reasoning_node(state: AgentState):
    text = state["extracted_text"]
    query = state["user_query"]

    if len(text.strip()) > 0:
        answer = qa_model(question=query, context=text)
        final_response = answer['answer']
    else:
        final_response = "Could not extract readable text to answer the query."

    return {"final_answer": final_response}

workflow = StateGraph(AgentState)
workflow.add_node("Preprocess", vision_preprocessing_node)
workflow.add_node("Layout", layout_analysis_node)
workflow.add_node("OCR", ocr_extraction_node)
workflow.add_node("Reasoning", llm_reasoning_node)

workflow.set_entry_point("Preprocess")
workflow.add_edge("Preprocess", "Layout")
workflow.add_edge("Layout", "OCR")
workflow.add_edge("OCR", "Reasoning")
workflow.add_edge("Reasoning", END)

multimodal_agent = workflow.compile()

if __name__ == "__main__":
    sample_url = "https://raw.githubusercontent.com/jaidedai/EasyOCR/master/examples/english.png"
    urllib.request.urlretrieve(sample_url, "sample_doc.png")

    initial_state = {
        "image_path": "sample_doc.png",
        "user_query": "What should I clean my hands with?"
    }

    print("Executing Multimodal Agent Workflow (Please wait for OCR and NLP inference)...\n")

    # Use invoke() to get the complete final state dictionary
    final_state = multimodal_agent.invoke(initial_state)

    img_bgr = cv2.imread(initial_state["image_path"])
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    for region in final_state["detected_regions"]:
        x1, y1, x2, y2 = region["bbox"]
        cv2.rectangle(img_rgb, (x1, y1), (x2, y2), (0, 255, 0), 3)

    plt.figure(figsize=(10, 8))
    plt.imshow(img_rgb)
    plt.title("Visual Perception: Layout Detection")
    plt.axis('off')
    plt.show()

    print("\n" + "="*60)
    print(f"Extracted Context (OCR): {final_state['extracted_text'][:120]}...")
    print(f"User Query: {initial_state['user_query']}")
    print(f"Agent Intelligent Answer: {final_state['final_answer']}")
    print("="*60)